# Fantasy Score Prediction — Exploration

In [ ]:
from models.database import SessionLocal
from ml.data.features import build_feature_matrix, build_today_features
from ml.models.predictor import FantasyPredictor

db = SessionLocal()
df = build_feature_matrix(db)
db.close()

print(f"{len(df)} rows, {df['player_id'].nunique()} players")
df.head()

In [ ]:
predictor = FantasyPredictor.load()

print("Feature importances:")
for feat, score in predictor.feature_importance().items():
    print(f"  {feat:<25} {score}")

In [ ]:
import matplotlib.pyplot as plt

scores = df["fantasy_score"].dropna()
min = scores.min()
max = scores.max()
print(f"Fantasy Score range: {min:.2f} to {max:.2f}")
bins = range(int(scores.min()), int(scores.max())+2)

plt.figure(figsize=(10, 3))
plt.hist(scores, bins=bins, color="blue")

plt.title("Distribution of Fantasy Scores")
plt.xlabel("Fantasy Score")
plt.ylabel("Frequency")

plt.show()


## Feature Correlation Analysis

In [ ]:
import pandas as pd
from scipy import stats

FEATURES_TO_CHECK = [
    "opp_def_rating",
    "opp_pace",
    "opp_ppg",
    "opp_rpg",
    "opp_apg",
    "is_home",
    "is_back_to_back",
    "fantasy_trend_5",
    "fantasy_trend_10",
    "std_fantasy_last_10",
    "avg_minutes_last_5",
    "games_last_10d",
]
TARGET = "fantasy_delta"

rows = []
for feat in FEATURES_TO_CHECK:
    sub = df[[feat, TARGET]].dropna()
    r, p = stats.pearsonr(sub[feat], sub[TARGET])
    rows.append({"feature": feat, "r": round(r, 4), "r²": round(r**2, 4), "p_value": round(p, 4), "significant": p < 0.05})

corr_df = pd.DataFrame(rows).sort_values("r²", ascending=False).reset_index(drop=True)
corr_df

## Tonight's Predictions

In [ ]:
from datetime import date

tonight = date(2026, 3, 13)
db = SessionLocal()
feat_df = build_today_features(db, tonight)
db.close()
print(f"Players with enough history to predict: {len(feat_df)}")
feat_df.head()

In [ ]:
predictor = FantasyPredictor.load()

feat_df["predicted_delta"] = predictor.predict(feat_df)
feat_df["predicted_fantasy"] = feat_df["avg_fantasy_season"] + feat_df["predicted_delta"]

results = (
    feat_df[["name", "team", "opponent", "is_home", "avg_fantasy_season", "fantasy_trend_5", "predicted_delta", "predicted_fantasy"]]
    .sort_values("predicted_fantasy", ascending=False)
    .reset_index(drop=True)
)
results.index += 1
for col in ["avg_fantasy_season", "fantasy_trend_5", "predicted_delta", "predicted_fantasy"]:
    results[col] = results[col].round(1)
results["matchup"] = results.apply(lambda r: f"{'vs' if r['is_home'] else '@'} {r['opponent']}", axis=1)

results[["name", "team", "matchup", "avg_fantasy_season", "fantasy_trend_5", "predicted_delta", "predicted_fantasy"]].head(20)